# 02 - Diffusion draft, AR verify

**학습 목표**: draft와 target을 왼쪽부터 비교해 longest matching prefix와 첫 AR correction을 commit하는 lossless greedy toy를 구현합니다. 실제 logits나 sampling rejection rule은 다루지 않습니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
def verify_round(target, start, draft):
    accepted = []
    for offset, proposed in enumerate(draft):
        position = start + offset
        if position >= len(target):
            break
        verifier_token = target[position]
        if proposed != verifier_token:
            # 첫 mismatch에는 verifier의 token을 넣어 반드시 전진합니다.
            accepted.append(verifier_token)
            return accepted, offset
        accepted.append(proposed)
    return accepted, None

target = 'diffusion drafts and autoregression verifies every accepted prefix'.split()
accepted, mismatch = verify_round(target, 0, ['diffusion', 'drafts', 'but', 'autoregression'])
print(accepted, 'mismatch offset=', mismatch)
assert accepted == ['diffusion', 'drafts', 'and']

In [ ]:
def make_draft(target, start, block_size, round_index):
    draft = target[start:start + block_size]
    # 짝수 round에서는 세 번째 제안을 일부러 틀리게 해 correction을 관찰합니다.
    if round_index % 2 == 0 and len(draft) >= 3:
        draft = list(draft)
        draft[2] = '<wrong>'
    return draft

output = []
rounds = 0
while len(output) < len(target):
    draft = make_draft(target, len(output), 4, rounds)
    committed, mismatch = verify_round(target, len(output), draft)
    output.extend(committed)
    rounds += 1
    print(f'round {rounds}: draft={draft}, commit={committed}, mismatch={mismatch}')

print('output:', ' '.join(output))
print('model forwards:', 2 * rounds)
assert output == target

틀린 draft는 최종 품질을 떨어뜨리지 않고 해당 round의 수락 길이를 줄입니다. Sampling 분포를 보존하려면 단순 argmax 비교가 아니라 정식 speculative rejection rule이 필요합니다.